# Open an example file

In [33]:
# Import modules
import os
import sys

# Make sure that the executable path exists in the environment path
env_bin = os.path.dirname(sys.executable)
if env_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{env_bin}:{os.environ['PATH']}"

import numpy as np
import pandas as pd
import xarray as xr
import yaml

with open("config.yml") as f:
    config = yaml.safe_load(f)
viz_opts = config["viz"]

import warnings

import matplotlib.pyplot as plt

# Suppress specific RuntimeWarning related to mean of empty slice
warnings.filterwarnings(action="ignore", category=RuntimeWarning, message=".*Mean of empty slice.*")

import os

for k in list(os.environ):
    if k.startswith(("PMI_", "PMIX_", "SLURM_MPI", "I_MPI_PMI")):
        os.environ.pop(k)

import pyproj
import xesmf as xe

In [34]:
# def setup_Weddell_Sea(fig):

#     # Define the projection for a high-resolution South Polar Stereographic map
#     high_res_proj = ccrs.SouthPolarStereo(central_longitude=-50)
#     high_res_proj.threshold = 1e3

#     ax = fig.add_subplot(111, projection=high_res_proj)
#     ax.add_feature(cfeature.LAND, color="grey")

#     # Define the limits for the x-axis and y-axis of the map
#     xlim = [-80, -20]  # longitude
#     ylim = [-60, -80]  # latitude

#     # Define a rectangular path for the map extent using matplotlib's Path object
#     rect = mpath.Path(
#         [
#             [xlim[0], ylim[0]],
#             [xlim[1], ylim[0]],
#             [xlim[1], ylim[1]],
#             [xlim[0], ylim[1]],
#             [xlim[0], ylim[0]],
#         ]
#     )

#     # Transform the rectangular path to the data coordinate system
#     proj_to_data = ccrs.PlateCarree()._as_mpl_transform(ax) - ax.transData
#     rect_in_target = proj_to_data.transform_path(rect)

#     # Set the boundary of the map using the transformed rectangular path
#     ax.set_boundary(rect_in_target)

#     # Configure gridlines for the map
#     gl = ax.gridlines(draw_labels=["top", "right", "x", "y"], rotate_labels=True, x_inline=False)

#     # Draw latitude and longitude gridlines (and their labels)
#     # only at the specified values
#     # lathelp = np.arange(-75, -60, 5)
#     # gl.ylocator = mticker.FixedLocator(lathelp)
#     # lonhelp = np.arange(-80, -10, 10)
#     # gl.xlocator = mticker.FixedLocator(lonhelp)

#     # Rotate the longitude labels to be more readable
#     gl.xlabel_style = {"rotation": 30}

#     # To make sure that all of the wanted map extent is visible in the figure,
#     # (especially because the chosen map projections differs strongly
#     # at the poles from the PlateCarree Projection)
#     # this hack of adding a constant is needed and may need be adjusted,
#     # dependent on your map extent and figure size
#     ax.set_extent([xlim[0], xlim[1], ylim[0] + 4, ylim[1]])

#     return ax

In [35]:
# fig = plt.figure(figsize=(8, 8))
# ax = setup_Weddell_Sea(fig)
# im = ax.pcolormesh(
#     ds.u.lon.values,
#     ds.u.lat.values,
#     np.squeeze(ds.u.values),
#     transform=ccrs.PlateCarree(),
#     cmap=cm.cm.thermal,
#     shading="nearest",
# )
# ax.add_feature(cfeature.LAND, color="lightgrey", alpha=0.9, zorder=10)
# # ax.contour(lon, lat, depth, transform=ccrs.PlateCarree(), levels = np.linspace(-5000,0,5), colors = "k", linestyles = "solid")
# plt.colorbar(im, orientation="vertical", shrink=0.4)
# for name, mooring in zip(mooring_names, list_of_moorings):
#     ax.plot(mooring.location.lon, mooring.location.lat, "k.", ms=2, transform=ccrs.PlateCarree())

In [36]:
def preprocess(ds):
    ds = ds.assign_coords(lon=ds.lon - 180)
    return ds.sel(lat=slice(-80, -50), lon=slice(-70, -30))

In [37]:
# documentation at https://www.cen.uni-hamburg.de/en/icdc/data/cryosphere/seaicedrift-satobs-global.html
# Importantly, we use north and east velocities, not the gridaligned velocities ux, vy
sea_ice_drift = xr.open_mfdataset(
    "./data/ice_drift/*/*.nc",
    combine="nested",
    concat_dim="time",
    drop_variables=["ux", "vy", "velocity", "sigma", "qualityflag_1", "qualityflag_2", "land"],
    chunks="auto",
    # preprocess=preprocess,
    data_vars="minimal",
    coords="minimal",
    compat="override",
)

In [38]:
sea_ice_drift

<xarray.Dataset> Size: 903MB
Dimensions:  (time: 1095, y: 321, x: 321)
Coordinates:
  * time     (time) datetime64[ns] 9kB 2017-01-01 2017-01-02 ... 2019-12-31
    lat      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    lon      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    u        (time, y, x) float32 451MB dask.array<chunksize=(1, 321, 321), meta=np.ndarray>
    v        (time, y, x) float32 451MB dask.array<chunksize=(1, 321, 321), meta=np.ndarray>
Attributes: (12/29)
    Conventions:                CF-1.6
    title:                      Daily gridded sea ice motion from combined Bu...
    source:                     Tschudi, M., W. N. Meier, J. S. Steward, C. F...
    history:                    Modified from source (see above) on Mon Apr 2...
    cdm_data_type:              EASE grid
    geographic_region:          polar regions
    ...                         ...
    geospatial_lon_resolution:  25km
    geospatial_lat_resolution:  25km
    platform:                   NOAA-Pathfinder NIMBUS DMSP
    sensor:                     AVHRR SMMR SSM/I SSMIS
    version:                    4.1
    citation:                   [1] Tschudi, M. A., Meier, W. N., and Stewart...

In [39]:
coarse_sea_ice_drift = sea_ice_drift.resample(
    time="120h", origin=pd.Timestamp("2017-02-12")
).mean()  # equivalent to 5 days

In [40]:
coarse_sea_ice_drift = coarse_sea_ice_drift.persist()
coarse_sea_ice_drift

<xarray.Dataset> Size: 182MB
Dimensions:  (time: 220, y: 321, x: 321)
Coordinates:
  * time     (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
    lat      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
    lon      (y, x) float32 412kB dask.array<chunksize=(321, 321), meta=np.ndarray>
Dimensions without coordinates: y, x
Data variables:
    u        (time, y, x) float32 91MB dask.array<chunksize=(1, 321, 321), meta=np.ndarray>
    v        (time, y, x) float32 91MB dask.array<chunksize=(1, 321, 321), meta=np.ndarray>
Attributes: (12/29)
    Conventions:                CF-1.6
    title:                      Daily gridded sea ice motion from combined Bu...
    source:                     Tschudi, M., W. N. Meier, J. S. Steward, C. F...
    history:                    Modified from source (see above) on Mon Apr 2...
    cdm_data_type:              EASE grid
    geographic_region:          polar regions
    ...                         ...
    geospatial_lon_resolution:  25km
    geospatial_lat_resolution:  25km
    platform:                   NOAA-Pathfinder NIMBUS DMSP
    sensor:                     AVHRR SMMR SSM/I SSMIS
    version:                    4.1
    citation:                   [1] Tschudi, M. A., Meier, W. N., and Stewart...

In [41]:
# coarse_sea_ice_drift.compute()

<xarray.Dataset> Size: 182MB
Dimensions:  (time: 220, y: 321, x: 321)
Coordinates:
  * time     (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
    lat      (y, x) float32 412kB -37.14 -37.31 -37.49 ... -37.49 -37.31 -37.14
    lon      (y, x) float32 412kB 225.0 224.8 224.6 224.5 ... 44.64 44.82 45.0
Dimensions without coordinates: y, x
Data variables:
    u        (time, y, x) float32 91MB nan nan nan nan nan ... nan nan nan nan
    v        (time, y, x) float32 91MB nan nan nan nan nan ... nan nan nan nan
Attributes: (12/29)
    Conventions:                CF-1.6
    title:                      Daily gridded sea ice motion from combined Bu...
    source:                     Tschudi, M., W. N. Meier, J. S. Steward, C. F...
    history:                    Modified from source (see above) on Mon Apr 2...
    cdm_data_type:              EASE grid
    geographic_region:          polar regions
    ...                         ...
    geospatial_lon_resolution:  25km
    geospatial_lat_resolution:  25km
    platform:                   NOAA-Pathfinder NIMBUS DMSP
    sensor:                     AVHRR SMMR SSM/I SSMIS
    version:                    4.1
    citation:                   [1] Tschudi, M. A., Meier, W. N., and Stewart...

In [67]:
# Conservative regridding
# with assistance by claude code

ds = coarse_sea_ice_drift

# EPSG:6933 = southpolar EASE-Grid 2.0
to_ll = pyproj.Transformer.from_crs("EPSG:6932", "EPSG:4326", always_xy=True)

# Cell centres -> 2D lon/lat
X, Y = np.meshgrid(ds.x.values, ds.y.values)
lon, lat = to_ll.transform(X, Y)

# Cell corners -> bounds (needed for conservative); shape (ny+1, nx+1)
dx = float(ds.x[1] - ds.x[0])
dy = float(ds.y[1] - ds.y[0])
xb = np.append(ds.x.values - dx / 2, ds.x.values[-1] + dx / 2)  # b for bounds
yb = np.append(ds.y.values - dy / 2, ds.y.values[-1] + dy / 2)  # b for bounds
Xb, Yb = np.meshgrid(xb, yb)
lon_b, lat_b = to_ll.transform(Xb, Yb)  # expected naming scheme for xesmf,

ds_in = ds.assign_coords(
    lon=(("y", "x"), lon),
    lat=(("y", "x"), lat),
    lon_b=(("y_b", "x_b"), lon_b),
    lat_b=(("y_b", "x_b"), lat_b),
).chunk({"x": -1, "y": -1})  # keep time chunked

# Centres at ..., -0.5, 0.0, 0.5, ...  (exact in floating point)
lon_center = np.arange(-360, 360) * 0.5  # -180.0 ... 179.5   (720 points)
lat_center = np.arange(-180, 181) * 0.5  # -90.0 ... 90.0     (361 points)

# Edges are centres +/- 0.25
lon_bounds = np.append(lon_center - 0.25, lon_center[-1] + 0.25)  # -180.25 ... 179.75 (721)
lat_bounds = np.append(lat_center - 0.25, lat_center[-1] + 0.25)  # -90.25 ... 90.25   (362)
lat_bounds = np.clip(lat_bounds, -90, 90)  # no edges beyond the poles

# Target 0.5° grid
ds_out = xr.Dataset(
    coords=dict(
        lon=("lon", lon_center),
        lat=("lat", lat_center),
        lon_b=("lon_b", lon_bounds),
        lat_b=("lat_b", lat_bounds),
    )
)

regridder = xe.Regridder(ds_in, ds_out, "conservative", unmapped_to_nan=True)
half_degree_coarse_sea_ice_drift = regridder(ds_in, keep_attrs=True)

In [68]:
half_degree_coarse_sea_ice_drift = (
    half_degree_coarse_sea_ice_drift.persist()
    .rename({"lat": "latitude", "lon": "longitude"})
    .fillna(0)
    .drop_vars(["lat_b", "lon_b"])
)
half_degree_coarse_sea_ice_drift

<xarray.Dataset> Size: 457MB
Dimensions:    (time: 220, latitude: 361, longitude: 720)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
  * latitude   (latitude) float64 3kB -90.0 -89.5 -89.0 -88.5 ... 89.0 89.5 90.0
  * longitude  (longitude) float64 6kB -180.0 -179.5 -179.0 ... 179.0 179.5
Data variables:
    u          (time, latitude, longitude) float32 229MB dask.array<chunksize=(1, 361, 720), meta=np.ndarray>
    v          (time, latitude, longitude) float32 229MB dask.array<chunksize=(1, 361, 720), meta=np.ndarray>
Attributes: (12/30)
    Conventions:                CF-1.6
    title:                      Daily gridded sea ice motion from combined Bu...
    source:                     Tschudi, M., W. N. Meier, J. S. Steward, C. F...
    history:                    Modified from source (see above) on Mon Apr 2...
    cdm_data_type:              EASE grid
    geographic_region:          polar regions
    ...                         ...
    geospatial_lat_resolution:  25km
    platform:                   NOAA-Pathfinder NIMBUS DMSP
    sensor:                     AVHRR SMMR SSM/I SSMIS
    version:                    4.1
    citation:                   [1] Tschudi, M. A., Meier, W. N., and Stewart...
    regrid_method:              conservative

In [69]:
# # collapse lat/lon to onedimensional coordinate axes
# half_degree_coarse_sea_ice_drift = (
#     half_degree_coarse_sea_ice_drift
#     .assign_coords(
#         y=half_degree_coarse_sea_ice_drift.lat.isel(x=0).values,
#         x=half_degree_coarse_sea_ice_drift.lon.isel(y=0).values,
#     )
#     .drop_vars(["lat", "lon","lat_b", "lon_b"])
#     .rename({"y": "latitude", "x": "longitude"})
#     .fillna(0)
# )

In [70]:
half_degree_coarse_sea_ice_drift = half_degree_coarse_sea_ice_drift.compute()
half_degree_coarse_sea_ice_drift

<xarray.Dataset> Size: 457MB
Dimensions:    (time: 220, latitude: 361, longitude: 720)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2016-12-29 2017-01-03 ... 2019-12-29
  * latitude   (latitude) float64 3kB -90.0 -89.5 -89.0 -88.5 ... 89.0 89.5 90.0
  * longitude  (longitude) float64 6kB -180.0 -179.5 -179.0 ... 179.0 179.5
Data variables:
    u          (time, latitude, longitude) float32 229MB 0.0 0.0 0.0 ... 0.0 0.0
    v          (time, latitude, longitude) float32 229MB 0.0 0.0 0.0 ... 0.0 0.0
Attributes: (12/30)
    Conventions:                CF-1.6
    title:                      Daily gridded sea ice motion from combined Bu...
    source:                     Tschudi, M., W. N. Meier, J. S. Steward, C. F...
    history:                    Modified from source (see above) on Mon Apr 2...
    cdm_data_type:              EASE grid
    geographic_region:          polar regions
    ...                         ...
    geospatial_lat_resolution:  25km
    platform:                   NOAA-Pathfinder NIMBUS DMSP
    sensor:                     AVHRR SMMR SSM/I SSMIS
    version:                    4.1
    citation:                   [1] Tschudi, M. A., Meier, W. N., and Stewart...
    regrid_method:              conservative

In [71]:
half_degree_coarse_sea_ice_drift.to_zarr("surface_stress_sea_ice_drift.zarr", mode="w")

/albedo/soft/sw/conda-sw/analysis-toolbox-python-04.2026/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


# Sanity Check

In [46]:
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.dates as mdates

In [47]:
cmap = plt.cm.viridis
number_of_moorings = len(list_of_moorings)
color_list = [cmap(i / number_of_moorings) for i in range(number_of_moorings)]
# color_list.append("xkcd:rose")
discrete_cmap = mcolors.LinearSegmentedColormap.from_list(
    "mooring_ids", color_list, N=number_of_moorings
)

NameError: name 'list_of_moorings' is not defined

In [ ]:
sea_ice_concentrations = pd.read_csv("./daily_sea_ice_concentrations.csv")  # , index_col = 0)
# sea_ice_concentrations.rename = "months"
sea_ice_concentrations.set_index(pd.to_datetime(sea_ice_concentrations["Unnamed: 0"]), inplace=True)
sea_ice_concentrations.drop("Unnamed: 0", inplace=True, axis="columns")

In [ ]:
# fig, ax = plt.subplots(1)
# ax2 = plt.twinx(ax)

fig, axis = plt.subplots(nrows=2, sharex=True)
ax, ax2 = axis

for mooring_id, col in enumerate(daily_sea_ice_drift.columns):
    ax.plot(daily_sea_ice_drift.index, daily_sea_ice_drift[col].abs(), color=color_list[mooring_id])

for mooring_id, col in enumerate(sea_ice_concentrations.columns):
    ax2.plot(
        sea_ice_concentrations.index,
        sea_ice_concentrations[col],
        color=color_list[mooring_id],
        ls="--",
    )

norm = mcolors.Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=discrete_cmap),
    ax=ax,
    location="top",
    orientation="horizontal",
)
cbar.ax.set_xticks(np.arange(0.5 / (number_of_moorings), 1, 1 / (number_of_moorings)))
cbar.ax.set_xticklabels(mooring_names)

ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
ax.tick_params(axis="x", which="both", labelsize=7)